In [1]:
l = int(input("Number of Slots: "))
t = int(input("Prime t: "))
m = int(input("Plaintext Modulus m: "))
messages = list(map(int, input("Enter messages (space-separated): ").split()))

F     = GF(t)
R     = F['X']
Phi_m = R(cyclotomic_polynomial(m))
omega = F.zeta(m)

# Primitive indices: k where gcd(k, m) = 1
primitive_indices = [k for k in range(1, m) if gcd(k, m) == 1]
primitive_roots = [omega**p for p in primitive_indices]
print(f"primitive indices: {primitive_indices}")
print(f"primitive_roots:{primitive_roots}")
# ── Forward NTT: coefficients → slot values ──────────────
def ntt(a, omega, t, m):
    """
    Evaluate polynomial f(X) = a_0 + a_1*X + ... at primitive m-th roots
    NTT(a)_k = sum_{j=0}^{l-1} a_j * omega^{k*j},  gcd(k,m)=1
    """
    F                = GF(t)
    primitive_indices = [k for k in range(1, m) if gcd(k, m) == 1]
    l                = len(a)
    return [sum(F(a[j]) * (omega^(k*j)) for j in range(l))
            for k in primitive_indices]

def intt_general(v, omega, t, m):
    """
    Correct INTT for ANY m — uses full Vandermonde inverse
    Works for both power-of-2 and general cyclotomic polynomials
    """
    F                 = GF(t)
    primitive_indices = [k for k in range(1, m) if gcd(k, m) == 1]
    l                 = len(v)

    # Primitive roots: omega_i = omega^{k_i}
    roots = [F(omega)^k for k in primitive_indices]

    # Build Vandermonde matrix V[i][j] = omega_i^j
    V = matrix(F, [[roots[i]^j for j in range(l)] for i in range(l)])

    # Compute V^{-1} exactly
    V_inv = V.inverse()

    # INTT = V^{-1} * v
    v_vec = vector(F, [F(vi) for vi in v])
    coeffs = V_inv * v_vec

    return list(coeffs)

# ── Encode: messages → polynomial f(X) via INTT ──────────
coeffs = intt_general(messages, omega, t, m)
print(f"\nINTT coefficients : {coeffs}")

f = R(coeffs) % Phi_m
print(f"\nEncoded polynomial:\nf(X) = {f}")
print(f"Degree            : {f.degree()}  (should be < phi(m) = {l})")

# ── Decode: f(X) → messages via NTT ──────────────────────
recovered = ntt(list(f), omega, t, m)

print(f"\nOriginal messages : {[F(x) for x in messages]}")
print(f"Recovered messages: {recovered}")
print(f"Correct           : {[F(x) for x in messages] == recovered}")

Number of Slots:  10

Prime t:  89

Plaintext Modulus m:  11

Enter messages (space-separated):  1 2 3 4 5 1 2 3 4 5 

primitive indices: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
primitive_roots:[64, 2, 39, 4, 78, 8, 67, 16, 45, 32]

INTT coefficients : [6, 6, 54, 66, 40, 29, 66, 55, 29, 41]

Encoded polynomial:
f(X) = 41*X^9 + 29*X^8 + 55*X^7 + 66*X^6 + 29*X^5 + 40*X^4 + 66*X^3 + 54*X^2 + 6*X + 6
Degree            : 9  (should be < phi(m) = 10)

Original messages : [1, 2, 3, 4, 5, 1, 2, 3, 4, 5]
Recovered messages: [1, 2, 3, 4, 5, 1, 2, 3, 4, 5]
Correct           : True
